In [ ]:
# @title 1️⃣ 系統環境初始化與降級偵測
import torch
import os

print("🔍 [1/2] 正在檢查運算環境...")
USE_CPU = not torch.cuda.is_available()

if USE_CPU:
    print("⚠️ 【警告】未偵測到 GPU！系統自動降級為純 CPU 模式。")
else:
    print(f"✅ 成功偵測到 GPU: {torch.cuda.get_device_name(0)}")

os.environ['COMFY_USE_CPU'] = 'true' if USE_CPU else 'false'

print("⚡ [2/2] 正在安裝極速下載神器 (aria2)...")
!apt-get -y install -qq aria2 > /dev/null
print("✅ 環境準備完畢！")

In [ ]:
# @title 2️⃣ Custom Nodes 模組化安裝區
import os
import subprocess

%cd /content
if not os.path.exists('/content/ComfyUI'):
    print("⏬ 正在 Clone ComfyUI 核心...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git > /dev/null

%cd /content/ComfyUI
print("📦 正在安裝 ComfyUI 核心依賴套件...")
!pip install -r requirements.txt -q

# 預先安裝底層庫 (避免 GGUF/ONNX 報錯)
print("📦 正在安裝底層運行庫 (ONNX, GGUF)...")
!pip install onnxruntime onnxruntime-gpu xformers -q

print("📦 正在同步 ComfyUI 自定義節點...")
NODE_DIR = "/content/ComfyUI/custom_nodes"
os.makedirs(NODE_DIR, exist_ok=True)

# 💡 未來只需在此陣列增刪 GitHub 連結
CUSTOM_NODES =[
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "https://github.com/yolain/ComfyUI-Easy-Use.git",
    # "https://github.com/yolain/ComfyUI-Easy-Sam3.git",
    # "https://github.com/city96/ComfyUI-GGUF.git",
    "https://github.com/rgthree/rgthree-comfy",
    # "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes",
    # "https://github.com/welltop-cn/ComfyUI-TeaCache",
]

for url in CUSTOM_NODES:
    repo_name = url.split('/')[-1].replace('.git', '')
    target_path = os.path.join(NODE_DIR, repo_name)

    if not os.path.exists(target_path):
        print(f"⬇️ 正在安裝: {repo_name}")
        subprocess.run(["git", "clone", url, target_path], check=True)
    else:
        print(f"⏩ 已存在，跳過: {repo_name}")

    # 🌟 新增：節點依賴自動安裝
    req_path = os.path.join(target_path, "requirements.txt")
    if os.path.exists(req_path):
        print(f"📦 安裝 {repo_name} 的專屬依賴...")
        # 隱藏冗長的下載日誌，但保留錯誤提示
        !pip install -r "{req_path}" -q

# 👑 終極防護：強制依賴鎖定 (Master Override)
# 無論上面的節點把版本改成什麼，最後一律強制對齊至最穩定的 1.x 終極版本，防止 SAM3 與其他節點衝突崩潰
print("🔨 正在執行底層依賴強制鎖定 (解決 Numpy/OpenCV 衝突)...")
!pip install numpy==1.26.4 opencv-python-headless==4.8.1.78 scikit-image -q 2>/dev/null
!pip install triton -q

print("✅ 節點與依賴同步完成！")

In [ ]:
# @title 3️⃣ 模型資源集中調度區 (GitHub 安全發布版)
import os
import subprocess
from google.colab import userdata

# ======================================================================
# 🔐 【安全設定區】
# 1. 請在 Colab 左側「鑰匙圖示 (Secrets)」加入 HF_Token
# 2. 本程式碼會自動從環境變數讀取，不會在 GitHub 留下字串紀錄
# ======================================================================
HF_TOKEN = userdata.get('HF_Token')

def download_model(url, folder, filename):
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)

    if not os.path.exists(filepath):
        print(f"⏳ 準備下載: {filename}...")

        # 這裡改用 list 形式傳給 subprocess，比 shell=True 更安全，防止 Token 洩漏到 log
        args = ['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M']

        if HF_TOKEN:
            args.append(f'--header=Authorization: Bearer {HF_TOKEN}')

        # 強制替換 /blob/ 為 /resolve/
        safe_url = url.replace("/blob/", "/resolve/")
        args.extend(['-d', folder, '-o', filename, safe_url])

        # 執行下載，並隱藏執行指令本身
        try:
            result = subprocess.run(args, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"✅ 下載完成: {filename}")
            else:
                print(f"❌ 下載失敗: {filename}")
                # 僅印出錯誤訊息後段，不印出包含 Token 的指令
        except Exception as e:
            print(f"⚠️ 執行錯誤: {filename}")
    else:
        print(f"⏩ 檔案已存在，跳過: {filename}")

# 📦 模型配置清單
# 註：此處網址建議保持為 main 分支之 resolve 連結
# 註：該模型託管於私人帳戶
MODEL_LIST = [
    {
        "url": "https://huggingface.co/JamesC9/Flux.2-Klein-TryOn/resolve/main/flux-2-klein-9b-fp8.safetensors",
        "folder": "/content/ComfyUI/models/diffusion_models",
        "filename": "flux-2-klein-9b-fp8.safetensors"
    },
    {
        "url": "https://huggingface.co/JamesC9/Flux.2-Klein-TryOn/resolve/main/qwen_3_8b_fp8mixed.safetensors",
        "folder": "/content/ComfyUI/models/text_encoders",
        "filename": "qwen_3_8b_fp8mixed.safetensors"
    },
    {
        "url": "https://huggingface.co/JamesC9/Flux.2-Klein-TryOn/resolve/main/flux2-vae.safetensors",
        "folder": "/content/ComfyUI/models/vae",
        "filename": "flux2-vae.safetensors"
    }
]

print("📥 正在向雲端節點請求模型資源...")
for m in MODEL_LIST:
    download_model(m["url"], m["folder"], m["filename"])
print("🎉 所有 AI 核心組件已部署完畢。")

In [ ]:
# @title 4️⃣ 雲端工作流 (UI & API JSON) 雙軌注入
import urllib.request
import urllib.error
import os
from google.colab import userdata

print("🗺️ 正在從 Hugging Face 載入工作流 JSON...")

# ======================================================================
# 🔐 若您的 Repo 為 Private，請填入 Token。若為 Public 則留空。
HF_TOKEN = userdata.get('HF_Token')
# ======================================================================

# 定義儲存路徑
UI_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(UI_DIR, exist_ok=True)

# 您的雙軌工作流配置 (請確保網址為 /resolve/main/...)
WORKFLOWS =[
    {
        "name": "🖥️ 前端 UI 編輯版",
        "url": "https://huggingface.co/JamesC9/Flux.2-Klein-TryOn/resolve/main/Klein_v1.5.json",
        # "url": "https://huggingface.co/JamesC9/Flux.2-Klein-TryOn/resolve/main/Klein_v1.6(Comparison).json",
        "path": os.path.join(UI_DIR, "Klein_v1.5.json")
    },
    {
        "name": "🤖 後端 API 呼叫版",
        "url": "https://huggingface.co/JamesC9/Flux.2-Klein-TryOn/resolve/main/API_Klein_v1.2.json",
        "path": "/content/ComfyUI/API_Klein_v1.2.json"
    }
]

def download_json(name, url, path):
    try:
        req = urllib.request.Request(url)
        if HF_TOKEN:
            req.add_header("Authorization", f"Bearer {HF_TOKEN}")

        with urllib.request.urlopen(req) as response, open(path, 'wb') as out_file:
            out_file.write(response.read())
        print(f"✅ 成功載入: {name} -> {path}")
    except urllib.error.HTTPError as e:
        print(f"❌ 下載失敗: {name} (HTTP {e.code}: 可能網址錯誤或缺乏 Token 權限)")
    except Exception as e:
        print(f"❌ 發生未預期錯誤: {name} - {e}")

# 執行下載
for wf in WORKFLOWS:
    download_json(wf["name"], wf["url"], wf["path"])

print("🎉 工作流配置完畢！LineBot 後端已可直接讀取根目錄的 API JSON。")

In [ ]:
# @title 5️⃣ 啟動伺服器與固定 API 網址 (Ngrok 極速修復版)
import os
import time
import subprocess
from google.colab import userdata

# ==========================================
# 🔐 【設定區】在此填入你的 Ngrok 資訊
# ==========================================
NGROK_TOKEN = userdata.get("NGROK_TOKEN")  # 貼上你的 Authtoken
NGROK_DOMAIN = userdata.get("NGROK_DOMAIN") # 替換為你的固定網域
# ==========================================

%cd /content/ComfyUI

print("🔧 正在極速安裝並設定 Ngrok...")
# 🚀 放棄耗時的 apt update，直接拉取二進位檔 (耗時 < 2秒)
!wget -q -c -nc https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz -C /usr/local/bin

# 設定 Authtoken (隱藏輸出以保持版面乾淨)
!ngrok config add-authtoken {NGROK_TOKEN} > /dev/null 2>&1

print("\n" + "🌟"*20)
print("🚀 【Ngrok 固定網址已啟用】")
print(f"👉 前端 UI 編輯網址: https://{NGROK_DOMAIN}")
print(f"🤖 LineBot API 端點: https://{NGROK_DOMAIN}/prompt")
print("🌟"*20 + "\n")

# 🚨 核心修復：強制 Ngrok 綁定 IPv4 (127.0.0.1:8188) 徹底解決 ERR_NGROK_8012 (IPv6 報錯)
subprocess.Popen(['ngrok', 'http', f'--domain={NGROK_DOMAIN}', '127.0.0.1:8188'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 給 Ngrok 2 秒鐘的建立通道時間
time.sleep(2)

print("🔥 正在啟動 ComfyUI 引擎 (即時日誌已開啟)...")

# 動態組合啟動指令，確保 Log 即時串流顯示
launch_cmd = "python main.py --use-pytorch-cross-attention --fp16-vae --fast"

if os.environ.get('COMFY_USE_CPU') == 'true':
    print("\n⚠️ 正在以【純 CPU 模式】強制啟動。生成速度將非常緩慢！")
    launch_cmd += " --cpu"

# 使用 Colab 的原生指令執行，確保 Log 直接顯示於下方
!{launch_cmd}